### Rag pipeline _ data ingestion to vectorDB pipeline


In [7]:
!python -m pip install langchain-text-splitters



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [13]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: SEC_4082_Web_App_Dev_Assignment_01.pdf
  ✓ Loaded 2 pages

Total documents loaded: 2


In [14]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 12.7.6 (Build 21H1320) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20261019061434Z00'00'", 'moddate': "D:20261019061434Z00'00'", 'source': '..\\data\\pdf\\SEC_4082_Web_App_Dev_Assignment_01.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SEC_4082_Web_App_Dev_Assignment_01.pdf', 'file_type': 'pdf'}, page_content="Web Engineering Assignment-01 \nHTML and CSS Fundamentals \nDEADLINE: 26TH OCT MONDAY, 2026 1ST LECTURE \nANY KIND OF CHEATING WILL RESULT IN 0. \nInstructions \n• Attempt all questions. Write your answers neatly on paper. \n• For code questions, write properly indented code. Marks are given for correct tags, \nattributes and syntax. \nSection A: Short Questions (20 marks) \nAnswer each question in 1 to 3 lines. Each question carries 2 marks. \n1. What is an unpaired (empty) tag? Give two examples.  [2] \n2. Write the line of code that starts an ordered list from the number 5.  [2] \n3. Which at

In [15]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [16]:
chunks=split_documents(all_pdf_documents)
chunks


Split 2 documents into 5 chunks

Example chunk:
Content: Web Engineering Assignment-01 
HTML and CSS Fundamentals 
DEADLINE: 26TH OCT MONDAY, 2026 1ST LECTURE 
ANY KIND OF CHEATING WILL RESULT IN 0. 
Instructions 
• Attempt all questions. Write your answers...
Metadata: {'producer': 'macOS Version 12.7.6 (Build 21H1320) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20261019061434Z00'00'", 'moddate': "D:20261019061434Z00'00'", 'source': '..\\data\\pdf\\SEC_4082_Web_App_Dev_Assignment_01.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SEC_4082_Web_App_Dev_Assignment_01.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 12.7.6 (Build 21H1320) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20261019061434Z00'00'", 'moddate': "D:20261019061434Z00'00'", 'source': '..\\data\\pdf\\SEC_4082_Web_App_Dev_Assignment_01.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SEC_4082_Web_App_Dev_Assignment_01.pdf', 'file_type': 'pdf'}, page_content="Web Engineering Assignment-01 \nHTML and CSS Fundamentals \nDEADLINE: 26TH OCT MONDAY, 2026 1ST LECTURE \nANY KIND OF CHEATING WILL RESULT IN 0. \nInstructions \n• Attempt all questions. Write your answers neatly on paper. \n• For code questions, write properly indented code. Marks are given for correct tags, \nattributes and syntax. \nSection A: Short Questions (20 marks) \nAnswer each question in 1 to 3 lines. Each question carries 2 marks. \n1. What is an unpaired (empty) tag? Give two examples.  [2] \n2. Write the line of code that starts an ordered list from the number 5.  [2] \n3. Which at